# RO-071 Pure pursuit: every number in the Note

The car, the course and the trackers live in `images/trackkit.py`: a kinematic bicycle with wheelbase 2.5 m and steering
at most 26.57° (tightest circle 5 m), the test course (10 m straight, a 90° left arc of radius 10 m, 10 m straight), the
start 1.0 m left of the line, trackers at 20 Hz and simulation steps of 0.01 s. This notebook runs each case and prints
the numbers the Note quotes. Nothing is random, so no seed is needed.

In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
import trackkit as tk
deg = np.degrees

def report(name, run, ref='rear'):
    m = tk.metrics(run, ref)
    print(f"{name:34s} within 0.1 m after {m['reach']:6.2f} m | past the line {m['over']:.3f} m | "
          f"arc: largest {m['arc_peak']:+.3f}, middle {m['arc_mid']:+.3f} | last straight {m['exit_peak']:+.3f} | "
          f"steering up to {m['steer_max']:.1f} deg | {run['t'][-1]:.2f} s")
    return m

## 1. The chased point and the arc at the start (Note, Sections 2 and 3)

In [2]:
x, y, th = tk.START
out = tk.pure_pursuit(x, y, th, tk.V_BASE)
print('chased point in the world:', round(out['gx'], 3), round(out['gy'], 3))
print('in the car frame: forward', round(out['cx'], 3), 'sideways', round(out['cy'], 3))
R = (out['cx']**2 + out['cy']**2) / (2 * out['cy'])
print('turning radius R =', round(R, 3), 'm; curvature', round(out['kappa'], 4), 'per m')
print('angle alpha =', round(deg(out['alpha']), 2), 'deg; 2 sin(alpha)/ld =', round(2*np.sin(out['alpha'])/4, 4))
print('steering =', round(deg(out['delta']), 2), 'deg')
print('check the triangle: 7^2 + 15 =', 7**2 + 15, '= 8^2')

chased point in the world: 3.873 0.0
in the car frame: forward 3.873 sideways -1.0
turning radius R = -8.0 m; curvature -0.125 per m
angle alpha = -14.48 deg; 2 sin(alpha)/ld = -0.125
steering = -17.35 deg
check the triangle: 7^2 + 15 = 64 = 8^2


## 2. The opening problem: RO-070's line rule on the arc (Note, Section 1)

RO-070's demonstration driver, RO-069's line rule at the nearest course point, run with RO-070's own kit
(`images/linekit.py`, a copy): turn rate -0.5 e - 2 th_e, steering atan(L omega / v).

In [3]:
import linekit as lk
line = lk.run(lambda q, t: lk.line_rule(q))
i = np.argmax(np.abs(line['e']) < 0.1)
print('line rule: within 0.1 m at s =', round(line['s'][i], 2), 'm')
arc = (line['s'] >= lk.L1) & (line['s'] <= lk.L1 + lk.L_ARC)
print('on the arc: largest error', round(line['e'][arc].min(), 3), 'm; middle of the arc',
      round(line['e'][np.argmin(np.abs(line['s'] - (lk.L1 + lk.L_ARC / 2)))], 3), 'm')
print("per metre at 2 m/s: e'' + 1.0 e' + 0.25 e = 0 -> natural frequency", np.sqrt(0.25), 'per m, damping ratio',
      1.0 / (2 * np.sqrt(0.25)))

line rule: within 0.1 m at s = 7.7 m
on the arc: largest error -0.388 m; middle of the arc -0.358 m
per metre at 2 m/s: e'' + 1.0 e' + 0.25 e = 0 -> natural frequency 0.5 per m, damping ratio 1.0


## 3. Pure pursuit with the 4 m look-ahead at 2 m/s (Note, Section 4)

In [4]:
run_pp = tk.simulate(tk.law('pp', ld=4.0), v=tk.V_BASE)
m_pp = report('pure pursuit 4 m, 2 m/s', run_pp)
ticks = run_pp['ticks']
print('ticks:', len(ticks), 'at 20 Hz; first tick steering', round(deg(ticks[0]['delta_cmd']), 2), 'deg')
# the error at the arc's entry, middle and exit
for s0 in (tk.S1, tk.S_MID, tk.S2):
    i = np.argmin(np.abs(run_pp['s'] - s0)); print(f'  s = {s0:.2f} m: error {run_pp["e"][i]:+.3f} m')

pure pursuit 4 m, 2 m/s            within 0.1 m after   7.36 m | past the line 0.000 m | arc: largest +0.120, middle +0.033 | last straight -0.166 | steering up to 17.4 deg | 17.86 s
ticks: 358 at 20 Hz; first tick steering -17.35 deg
  s = 10.00 m: error +0.065 m
  s = 17.85 m: error +0.033 m
  s = 25.71 m: error -0.092 m


In [5]:
# Figure 4's frames: the arc the car is told to drive at 3.0 s and at 10 s
for t0 in (3.0, 10.0):
    tick = ticks[int(round(t0 * tk.RATE))]
    i = int(round(tick['t'] / tk.DT))
    side = 'left' if tick['kappa'] > 0 else 'right'
    print(f"t = {tick['t']:.1f} s: lateral error {run_pp['e'][i]:+.2f} m, arc radius {abs(1 / tick['kappa']):.1f} m "
          f"(centre to the {side}), steering {deg(tick['delta_cmd']):+.1f} deg")

t = 3.0 s: lateral error +0.23 m, arc radius 37.3 m (centre to the left), steering +3.8 deg
t = 10.0 s: lateral error +0.01 m, arc radius 9.7 m (centre to the left), steering +14.4 deg


## 4. Why pure pursuit does not weave: the look-ahead point's sideways distance (Note, Section 4.2)

Near a straight line, with lateral error e and heading error th_e, the look-ahead point sits about e + ld th_e to the
side, so the curvature is -(2/ld^2) e - (2/ld) th_e: a spring on the lateral error and a damper on its rate per metre.

In [6]:
ld, e0, the0 = 4.0, 0.5, 0.1
print('sideways distance of the look-ahead point ~', -(e0 + ld * the0), 'm; curvature ~', round(-2*(e0 + ld*the0)/ld**2, 4))
# exact check with the kit: car at (0, 0.5) heading 0.1 rad
o = tk.pure_pursuit(0.0, 0.5, 0.1, 2.0, ld=4.0)
print('exact: sideways distance', round(o['cy'], 4), 'curvature', round(o['kappa'], 4))
wn = np.sqrt(2) / ld; zeta = (2 / ld) / (2 * wn)
print('natural frequency', round(wn, 4), 'per m; damping ratio', round(zeta, 4))
print('overshoot exp(-pi zeta / sqrt(1 - zeta^2)) =', round(np.exp(-np.pi*zeta/np.sqrt(1 - zeta**2)), 4))
print('first peak past the line at s = pi ld =', round(np.pi * ld, 2), 'm for ld = 4;', round(np.pi * 2, 2), 'm for ld = 2')

sideways distance of the look-ahead point ~ -0.9 m; curvature ~ -0.1125
exact: sideways distance -0.8937 curvature -0.1117
natural frequency 0.3536 per m; damping ratio 0.7071
overshoot exp(-pi zeta / sqrt(1 - zeta^2)) = 0.0432
first peak past the line at s = pi ld = 12.57 m for ld = 4; 6.28 m for ld = 2


## 5. The look-ahead sweep on the plan's car, and speed (Note, Section 5.1)

In [7]:
for v in (2.0, 5.0):
    for ld in (2.0, 4.0, 8.0):
        report(f'ld {ld:.0f} m, {v:.0f} m/s, no lag', tk.simulate(tk.law('pp', ld=ld), v=v))

ld 2 m, 2 m/s, no lag              within 0.1 m after   4.02 m | past the line 0.042 m | arc: largest +0.040, middle -0.002 | last straight -0.040 | steering up to 26.6 deg | 17.92 s


ld 4 m, 2 m/s, no lag              within 0.1 m after   7.36 m | past the line 0.000 m | arc: largest +0.120, middle +0.033 | last straight -0.166 | steering up to 17.4 deg | 17.86 s


ld 8 m, 2 m/s, no lag              within 0.1 m after  23.88 m | past the line 0.000 m | arc: largest +0.850, middle +0.652 | last straight -0.677 | steering up to 14.2 deg | 17.48 s


ld 2 m, 5 m/s, no lag              within 0.1 m after   3.93 m | past the line 0.042 m | arc: largest +0.038, middle -0.001 | last straight -0.037 | steering up to 26.6 deg | 7.17 s


ld 4 m, 5 m/s, no lag              within 0.1 m after   7.24 m | past the line 0.000 m | arc: largest +0.114, middle +0.030 | last straight -0.160 | steering up to 17.4 deg | 7.15 s


ld 8 m, 5 m/s, no lag              within 0.1 m after  23.85 m | past the line 0.000 m | arc: largest +0.833, middle +0.634 | last straight -0.667 | steering up to 14.3 deg | 7.00 s


## 6. A real steering motor: first-order lag of 0.4 s (Note, Section 5.2)

Stanford's car modelled its steering as a first-order lag with tau = 0.4 s (Hoffmann et al. 2007, eq. 4).

In [8]:
TAU = 0.4
sweep = {}
for ld in (2.0, 4.0, 8.0):
    sweep[ld] = report(f'ld {ld:.0f} m, 5 m/s, lag 0.4 s', tk.simulate(tk.law('pp', ld=ld), v=tk.V_FAST, tau=TAU))
for ld in (2.0, 4.0, 8.0):
    report(f'ld {ld:.0f} m, 2 m/s, lag 0.4 s', tk.simulate(tk.law('pp', ld=ld), v=tk.V_BASE, tau=TAU))
print('lag as distance: 2 m/s ->', 2 * TAU, 'm; 5 m/s ->', 5 * TAU, 'm')
print('wheels after one time constant, command -17.35 deg:', round((1 - np.exp(-1)) * -17.354, 2), 'deg')
o2 = tk.pure_pursuit(*tk.START, 2.0, ld=2.0)
print('2 m look-ahead at the start: curvature', round(o2['kappa'], 3), 'asks', round(deg(o2['delta']), 2), 'deg')

ld 2 m, 5 m/s, lag 0.4 s           within 0.1 m after   4.56 m | past the line 0.832 m | arc: largest -1.254, middle -0.787 | last straight -0.822 | steering up to 26.3 deg | 7.53 s
ld 4 m, 5 m/s, lag 0.4 s           within 0.1 m after   6.41 m | past the line 0.256 m | arc: largest -0.240, middle +0.018 | last straight +0.184 | steering up to 14.3 deg | 7.19 s


ld 8 m, 5 m/s, lag 0.4 s           within 0.1 m after  17.30 m | past the line 0.000 m | arc: largest +0.559, middle +0.058 | last straight -0.434 | steering up to 15.7 deg | 7.12 s


ld 2 m, 2 m/s, lag 0.4 s           within 0.1 m after   3.98 m | past the line 0.212 m | arc: largest +0.078, middle +0.010 | last straight +0.026 | steering up to 24.9 deg | 17.96 s


ld 4 m, 2 m/s, lag 0.4 s           within 0.1 m after   6.59 m | past the line 0.037 m | arc: largest -0.057, middle -0.001 | last straight -0.098 | steering up to 14.5 deg | 17.89 s


ld 8 m, 2 m/s, lag 0.4 s           within 0.1 m after  22.86 m | past the line 0.000 m | arc: largest +0.707, middle +0.449 | last straight -0.569 | steering up to 14.7 deg | 17.59 s
lag as distance: 2 m/s -> 0.8 m; 5 m/s -> 2.0 m
wheels after one time constant, command -17.35 deg: -10.97 deg
2 m look-ahead at the start: curvature -0.5 asks -51.34 deg


## 7. A look-ahead that grows with speed: ld = v x 2 s (Note, Section 6)

In [9]:
for v in (2.0, 5.0):
    print(f'{v} m/s: look-ahead', np.clip(v * 2.0, 2, 10), 'm')
    report(f'ld = v x 2 s, {v:.0f} m/s, lag', tk.simulate(tk.law('pp', ld_time=2.0), v=v, tau=TAU))
# Autoware's rule (pure_pursuit.param.yaml): clamp(2.4 v - 120 |kappa| + 3.6 |e| if |e| >= 0.5, 4.35, 15)
def autoware_ld(v, kappa=0.0, e=0.0):
    ld = 2.4 * abs(v) - 120 * abs(kappa) + (3.6 * abs(e) if abs(e) >= 0.5 else 0.0)
    return float(np.clip(ld, 4.35, 15.0))
for v, k, e in ((2, 0, 0), (5, 0, 0), (5, 0.1, 0), (5, 0, 1.0), (10, 0, 0)):
    print(f'Autoware look-ahead at {v} m/s, curvature {k}, lateral error {e}: {autoware_ld(v, k, e):.2f} m')

2.0 m/s: look-ahead 4.0 m


ld = v x 2 s, 2 m/s, lag           within 0.1 m after   6.59 m | past the line 0.037 m | arc: largest -0.057, middle -0.001 | last straight -0.098 | steering up to 14.5 deg | 17.89 s
5.0 m/s: look-ahead 10.0 m
ld = v x 2 s, 5 m/s, lag           within 0.1 m after  22.55 m | past the line 0.000 m | arc: largest +0.978, middle +0.604 | last straight -0.755 | steering up to 14.4 deg | 7.01 s
Autoware look-ahead at 2 m/s, curvature 0, lateral error 0: 4.80 m
Autoware look-ahead at 5 m/s, curvature 0, lateral error 0: 12.00 m
Autoware look-ahead at 5 m/s, curvature 0.1, lateral error 0: 4.35 m
Autoware look-ahead at 5 m/s, curvature 0, lateral error 1.0: 15.00 m
Autoware look-ahead at 10 m/s, curvature 0, lateral error 0: 15.00 m


## 8. Regulated pure pursuit: slow down on tight curves (Note, Section 7)

In [10]:
run_fast = tk.simulate(tk.law('pp', ld_time=2.0), v=tk.V_FAST, tau=TAU)
run_rpp = tk.simulate(tk.law('rpp', r_min=25.0, ld_time=2.0), v=tk.V_FAST, tau=TAU, a_max=2.5)
run_slow = tk.simulate(tk.law('pp', ld=4.0), v=tk.V_BASE, tau=TAU)
report('5 m/s, ld = v x 2 s', run_fast); report('regulated, r_min 25 m', run_rpp); report('2 m/s, ld 4 m', run_slow)
v = run_rpp['v']; s = run_rpp['s']
for name, r in (('5 m/s', run_fast), ('regulated', run_rpp)):
    i = np.argmin(np.abs(r['s'] - tk.S1)); print(name, ': lateral error at the arc entry', round(r['e'][i], 3))
i = np.argmin(np.abs(s - tk.S_MID)); print('regulated at the middle of the arc:', round(v[i], 2), 'm/s')
print('regulated: slowest', round(v.min(), 2), 'm/s at s =', round(s[np.argmin(v)], 2), 'm; starts slowing at s =',
      round(s[np.argmax(v < 4.99)], 2), 'm; look-ahead there', round(np.clip(v.min() * 2, 2, 10), 2), 'm')
print('speed on the 10 m arc if the commanded arc matched it: 5 x 10 / 25 =', 5 * 10 / 25, 'm/s')
print('lateral acceleration on the arc: 5 m/s', 25 / 10, 'm/s^2; 2 m/s', 4 / 10, 'm/s^2')
# Nav2's proximity rule: v * d / d_prox when closer than d_prox (cost_scaling_dist 0.6 m, gain 1.0)
print('Nav2 proximity: obstacle 0.3 m away ->', 0.5 * 0.3 / 0.6, 'm/s from 0.5 m/s')

5 m/s, ld = v x 2 s                within 0.1 m after  22.55 m | past the line 0.000 m | arc: largest +0.978, middle +0.604 | last straight -0.755 | steering up to 14.4 deg | 7.01 s
regulated, r_min 25 m              within 0.1 m after  19.31 m | past the line 0.000 m | arc: largest +0.948, middle +0.271 | last straight -0.142 | steering up to 19.3 deg | 11.36 s
2 m/s, ld 4 m                      within 0.1 m after   6.59 m | past the line 0.037 m | arc: largest -0.057, middle -0.001 | last straight -0.098 | steering up to 14.5 deg | 17.89 s
5 m/s : lateral error at the arc entry 0.908
regulated : lateral error at the arc entry 0.901
regulated at the middle of the arc: 1.49 m/s
regulated: slowest 1.41 m/s at s = 18.97 m; starts slowing at s = 7.05 m; look-ahead there 2.81 m
speed on the 10 m arc if the commanded arc matched it: 5 x 10 / 25 = 2.0 m/s
lateral acceleration on the arc: 5 m/s 2.5 m/s^2; 2 m/s 0.4 m/s^2
Nav2 proximity: obstacle 0.3 m away -> 0.25 m/s from 0.5 m/s


## 9. On a long arc any look-ahead below 2R gives the arc's curvature (Coulter 1992, §4.2)

In [11]:
for ld in (2.0, 4.0, 8.0):
    # rear axle on the arc 1 m after it starts (Note, Sections 4.1 and 5.1), heading along it; the look-ahead point is still on the arc
    xs, ys, hs, _ = tk.course(tk.S1 + 1.0)
    o = tk.pure_pursuit(float(xs), float(ys), float(hs), 2.0, ld=ld)
    print(f'ld {ld:4.1f} m: curvature {o["kappa"]:.4f} per m (the arc: 0.1)')

ld  2.0 m: curvature 0.1000 per m (the arc: 0.1)
ld  4.0 m: curvature 0.1000 per m (the arc: 0.1)
ld  8.0 m: curvature 0.1000 per m (the arc: 0.1)
